In [1]:
# Spanish Dialect Text Classifier (TF-IDF, SVM, BERT)

import pandas as pd
import numpy as np
import torch
import evaluate
from datasets import Dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

# Load data
files = [
    ("data/spain_female_1.parquet", "spain"),
    ("data/spain_female_2.parquet", "spain"),
    ("data/spain_male_1.parquet", "spain"),
    ("data/spain_male_2.parquet", "spain"),
    ("data/mexico_female.parquet", "mexico"),
    ("data/mexico_male.parquet", "mexico"),
]
dfs = [pd.read_parquet(f).assign(label=label) for f, label in files]
df = pd.concat(dfs).reset_index(drop=True)

#  Label encoding
label2id = {"spain": 0, "mexico": 1}
id2label = {v: k for k, v in label2id.items()}
df["label"] = df["label"].map(label2id)

# Stratified 80/20 split
train_texts, test_texts = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)
train_texts = train_texts.reset_index(drop=True)
test_texts = test_texts.reset_index(drop=True)

# Check balance
print(train_texts["label"].value_counts())
print(test_texts["label"].value_counts())


label
0    349
1    281
Name: count, dtype: int64
label
0    87
1    71
Name: count, dtype: int64


In [2]:

# TF-IDF + Logistic Regression
vectorizer = TfidfVectorizer(max_features=3000)
X_train = vectorizer.fit_transform(train_texts["text"])
X_test = vectorizer.transform(test_texts["text"])

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, train_texts["label"])

preds = clf.predict(X_test)
print("\n Logistic Regression:")
print(classification_report(test_texts["label"], preds, target_names=label2id.keys()))


 Logistic Regression:
              precision    recall  f1-score   support

       spain       0.80      0.97      0.88        87
      mexico       0.94      0.70      0.81        71

    accuracy                           0.85       158
   macro avg       0.87      0.83      0.84       158
weighted avg       0.86      0.85      0.84       158



In [3]:

# SVM
svm = SVC(kernel="linear")
svm.fit(X_train, train_texts["label"])

svm_preds = svm.predict(X_test)
print("\n SVM:")
print(classification_report(test_texts["label"], svm_preds, target_names=label2id.keys()))


 SVM:
              precision    recall  f1-score   support

       spain       0.91      0.95      0.93        87
      mexico       0.94      0.89      0.91        71

    accuracy                           0.92       158
   macro avg       0.93      0.92      0.92       158
weighted avg       0.92      0.92      0.92       158



In [4]:
# BERT
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import classification_report
from datasets import Dataset
import numpy as np
import evaluate
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model and tokenizer
model_ckpt = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

# Tokenization
def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True)

train_ds = Dataset.from_pandas(train_texts, preserve_index=False).map(tokenize_fn, batched=True)
test_ds = Dataset.from_pandas(test_texts, preserve_index=False).map(tokenize_fn, batched=True)

train_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# Model
model = AutoModelForSequenceClassification.from_pretrained(
    model_ckpt,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
).to(device)

# TrainingArguments
args = TrainingArguments(
    output_dir="bert-dialect-output",
    evaluation_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    learning_rate=2e-5,
    logging_steps=10,
    report_to="none",
    load_best_model_at_end=True,
    save_strategy="epoch",
    metric_for_best_model="eval_accuracy",
    greater_is_better=True
)


# Metric function with detailed report
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    report = classification_report(
        labels,
        preds,
        target_names=["spain", "mexico"],
        output_dict=True
    )
    return {
        "accuracy": report["accuracy"],
        "precision_spain": report["spain"]["precision"],
        "recall_spain": report["spain"]["recall"],
        "f1_spain": report["spain"]["f1-score"],
        "precision_mexico": report["mexico"]["precision"],
        "recall_mexico": report["mexico"]["recall"],
        "f1_mexico": report["mexico"]["f1-score"],
        "macro_f1": report["macro avg"]["f1-score"],
        "weighted_f1": report["weighted avg"]["f1-score"]
    }

# Trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

print("\n Fine-tuning BERT...")
trainer.train()

print("\n BERT Results:")
print(trainer.evaluate())


C:\Users\swift\PycharmProjects\Spanish_Clean\.venv\Lib\site-packages\huggingface_hub\file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Map:   0%|          | 0/630 [00:00<?, ? examples/s]

Map:   0%|          | 0/158 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\swift\PycharmProjects\Spanish_Clean\.venv\Lib\site-packages\accelerate\accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(



 Fine-tuning BERT...


Epoch,Training Loss,Validation Loss,Accuracy,Precision Spain,Recall Spain,F1 Spain,Precision Mexico,Recall Mexico,F1 Mexico,Macro F1,Weighted F1
1,0.587400,0.383258,0.835443,0.790476,0.954023,0.864583,0.924528,0.690141,0.790323,0.827453,0.831213
2,0.308100,0.391813,0.898734,0.927711,0.885057,0.905882,0.866667,0.915493,0.890411,0.898147,0.898930
3,0.144000,0.467163,0.905063,0.909091,0.919540,0.914286,0.900000,0.887324,0.893617,0.903951,0.904998
4,0.083900,0.599014,0.886076,0.948052,0.839080,0.890244,0.827160,0.943662,0.881579,0.885911,0.886350
5,0.038400,0.585114,0.905063,0.950000,0.873563,0.910180,0.858974,0.943662,0.899329,0.904754,0.905304
6,0.000500,0.495978,0.917722,0.911111,0.942529,0.926554,0.926471,0.887324,0.906475,0.916514,0.917531
7,0.000400,0.531293,0.917722,0.920455,0.931034,0.925714,0.914286,0.901408,0.907801,0.916758,0.917665
8,0.000300,0.617385,0.917722,0.940476,0.908046,0.923977,0.891892,0.929577,0.910345,0.917161,0.917851
9,0.000300,0.610269,0.911392,0.939759,0.896552,0.917647,0.880000,0.929577,0.904110,0.910878,0.911564
10,0.000300,0.613199,0.911392,0.939759,0.896552,0.917647,0.880000,0.929577,0.904110,0.910878,0.911564



 BERT Results:


{'eval_loss': 0.4959777593612671, 'eval_accuracy': 0.9177215189873418, 'eval_precision_spain': 0.9111111111111111, 'eval_recall_spain': 0.9425287356321839, 'eval_f1_spain': 0.9265536723163842, 'eval_precision_mexico': 0.9264705882352942, 'eval_recall_mexico': 0.8873239436619719, 'eval_f1_mexico': 0.9064748201438849, 'eval_macro_f1': 0.9165142462301346, 'eval_weighted_f1': 0.917530896973046, 'eval_runtime': 1.9727, 'eval_samples_per_second': 80.095, 'eval_steps_per_second': 10.139, 'epoch': 10.0}
